# 01 · Shared Training & Evaluation Pipeline (Hannah)

Reusable code used by **every** deep model: seeding, optimizer/scheduler builders, a training loop with early stopping and checkpointing, and an evaluation function that returns accuracy, macro-F1, per-class precision/recall/F1, and the confusion matrix.

Model selection uses **validation macro-F1** (the project's primary metric).

At the bottom is a smoke test on random data, so the pipeline can be verified before Alex's dataset is ready.

In [ ]:
# --- Locate the project root so `src/` imports work from any folder ---
import sys, os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
%%writefile src/pipeline.py
import copy
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

from src.config import TrainConfig, CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR, get_device


# ----------------------------------------------------------------------------
# Reproducibility
# ----------------------------------------------------------------------------
def set_seed(seed: int = 42, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


# ----------------------------------------------------------------------------
# Optimizer / scheduler
# ----------------------------------------------------------------------------
def build_optimizer(params, cfg: TrainConfig):
    params = [p for p in params if p.requires_grad]
    name = cfg.optimizer.lower()
    if name == "adamw":
        return torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    if name == "adam":
        return torch.optim.Adam(params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    if name == "sgd":
        return torch.optim.SGD(params, lr=cfg.lr, momentum=0.9, nesterov=True,
                               weight_decay=cfg.weight_decay)
    raise ValueError(f"Unknown optimizer: {cfg.optimizer}")


def build_scheduler(optimizer, cfg: TrainConfig):
    name = cfg.scheduler.lower()
    if name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(cfg.epochs, 1))
    if name == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(cfg.epochs // 3, 1), gamma=0.1)
    if name == "none":
        return None
    raise ValueError(f"Unknown scheduler: {cfg.scheduler}")


# ----------------------------------------------------------------------------
# One training epoch
# ----------------------------------------------------------------------------
def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None, grad_clip=None):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    use_amp = scaler is not None

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(x)
            loss = criterion(out, y)

        if use_amp:
            scaler.scale(loss).backward()
            if grad_clip:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        total_loss += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        n += y.size(0)

    return total_loss / n, correct / n


# ----------------------------------------------------------------------------
# Evaluation: accuracy, macro F1, per-class precision/recall, confusion matrix
# ----------------------------------------------------------------------------
@torch.no_grad()
def evaluate(model, loader, device, num_classes: int = 43, criterion=None):
    model.eval()
    criterion = criterion or nn.CrossEntropyLoss()
    all_preds, all_targets = [], []
    total_loss, n = 0.0, 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        out = model(x)
        total_loss += criterion(out, y).item() * y.size(0)
        n += y.size(0)
        all_preds.append(out.argmax(1).cpu())
        all_targets.append(y.cpu())

    preds = torch.cat(all_preds).numpy()
    targets = torch.cat(all_targets).numpy()
    labels = list(range(num_classes))
    p, r, f, s = precision_recall_fscore_support(
        targets, preds, labels=labels, average=None, zero_division=0)

    return {
        "loss": total_loss / n,
        "accuracy": float((preds == targets).mean()),
        "macro_precision": float(p.mean()),
        "macro_recall": float(r.mean()),
        "macro_f1": float(f.mean()),
        "per_class": {
            "precision": p.tolist(), "recall": r.tolist(),
            "f1": f.tolist(), "support": s.tolist(),
        },
        "confusion_matrix": confusion_matrix(targets, preds, labels=labels),
        "preds": preds,
        "targets": targets,
    }


def _scalars(m):
    return {k: m[k] for k in ("loss", "accuracy", "macro_precision", "macro_recall", "macro_f1")}


# ----------------------------------------------------------------------------
# Full training loop (early stopping on val macro-F1, best-checkpoint saving)
# ----------------------------------------------------------------------------
def fit(model, train_loader, val_loader, cfg: TrainConfig, run_name: str,
        device=None, restore_best: bool = True):
    device = device or get_device()
    model.to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    optimizer = build_optimizer(model.parameters(), cfg)
    scheduler = build_scheduler(optimizer, cfg)
    use_amp = cfg.amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    history = {k: [] for k in
               ["train_loss", "train_acc", "val_loss", "val_acc", "val_macro_f1", "lr", "epoch_time"]}
    best_f1, best_epoch, best_state, bad_epochs = -1.0, 0, None, 0
    ckpt_path = CHECKPOINT_DIR / f"{run_name}.pt"

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer,
                                          device, scaler, cfg.grad_clip)
        val = evaluate(model, val_loader, device, cfg.num_classes)
        lr_now = optimizer.param_groups[0]["lr"]
        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(val["loss"])
        history["val_acc"].append(val["accuracy"])
        history["val_macro_f1"].append(val["macro_f1"])
        history["lr"].append(lr_now)
        history["epoch_time"].append(time.time() - t0)

        print(f"[{run_name}] epoch {epoch:02d}/{cfg.epochs} | "
              f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {val['loss']:.4f} acc {val['accuracy']:.4f} "
              f"macroF1 {val['macro_f1']:.4f} | lr {lr_now:.2e} | {time.time() - t0:.1f}s")

        if val["macro_f1"] > best_f1:
            best_f1, best_epoch, bad_epochs = val["macro_f1"], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save({"model_state": best_state, "cfg": cfg.to_dict(),
                        "epoch": epoch, "val": _scalars(val)}, ckpt_path)
        else:
            bad_epochs += 1
            if bad_epochs >= cfg.early_stop_patience:
                print(f"[{run_name}] early stopping at epoch {epoch} (best epoch {best_epoch})")
                break

    if restore_best and best_state is not None:
        model.load_state_dict(best_state)

    history["best_epoch"] = best_epoch
    history["best_val_macro_f1"] = best_f1
    (RESULTS_DIR / f"{run_name}_history.json").write_text(json.dumps(history, indent=2))
    return history


# ----------------------------------------------------------------------------
# Utilities
# ----------------------------------------------------------------------------
def load_checkpoint(model, run_name: str, device=None):
    device = device or get_device()
    ckpt = torch.load(CHECKPOINT_DIR / f"{run_name}.pt", map_location=device)
    model.load_state_dict(ckpt["model_state"])
    return model.to(device).eval(), ckpt


def save_run_summary(run_name: str, cfg: TrainConfig, metrics: dict, split: str = "val", extra=None):
    """Writes metrics to results/<run>_<split>_metrics.json (used later for the final tables)."""
    out = {"run": run_name, "split": split, "cfg": cfg.to_dict(), **_scalars(metrics),
           "per_class": metrics["per_class"],
           "confusion_matrix": metrics["confusion_matrix"].tolist()}
    if extra:
        out.update(extra)
    (RESULTS_DIR / f"{run_name}_{split}_metrics.json").write_text(json.dumps(out, indent=2))


def plot_history(history: dict, title: str = "run", save: bool = True):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
    ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].legend()
    ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="val")
    ax[1].set_title("Accuracy"); ax[1].legend()
    ax[2].plot(history["val_macro_f1"], color="tab:green"); ax[2].set_title("Val macro-F1")
    for a in ax:
        a.set_xlabel("epoch"); a.grid(alpha=0.3)
    fig.suptitle(title)
    plt.tight_layout()
    if save:
        fig.savefig(FIGURES_DIR / f"{title}_history.png", dpi=150)
    plt.show()


def make_fake_loaders(cfg: TrainConfig, n_train=512, n_val=128, n_test=128):
    """RANDOM DATA - only for smoke-testing code before Alex's GTSRB loaders exist.
    Results from these loaders are meaningless."""
    g = torch.Generator().manual_seed(cfg.seed)

    def _mk(n, shuffle):
        x = torch.rand(n, cfg.in_channels, cfg.img_size, cfg.img_size, generator=g)
        y = torch.randint(0, cfg.num_classes, (n,), generator=g)
        return DataLoader(TensorDataset(x, y), batch_size=cfg.batch_size, shuffle=shuffle)

    return _mk(n_train, True), _mk(n_val, False), _mk(n_test, False)

## Smoke test (random data + a trivial linear model)

This only checks that the loop, early stopping, metrics, checkpointing, and plots work. The tiny model is a throwaway placeholder, not one of the project models.

In [ ]:
import torch.nn as nn
from src.config import TrainConfig, CHECKPOINT_DIR, RESULTS_DIR, get_device
from src.pipeline import (set_seed, fit, evaluate, plot_history,
                          make_fake_loaders, load_checkpoint, save_run_summary)

cfg = TrainConfig(epochs=3, batch_size=64, num_workers=0)
set_seed(cfg.seed)
device = get_device()

train_loader, val_loader, test_loader = make_fake_loaders(cfg)
model = nn.Sequential(nn.Flatten(),
                      nn.Linear(cfg.in_channels * cfg.img_size**2, cfg.num_classes))

history = fit(model, train_loader, val_loader, cfg, run_name="smoke_test", device=device)
plot_history(history, title="smoke_test", save=False)

metrics = evaluate(model, val_loader, device, cfg.num_classes)
print({k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})
print("Confusion matrix shape:", metrics["confusion_matrix"].shape)

model2, ckpt = load_checkpoint(nn.Sequential(nn.Flatten(),
                               nn.Linear(cfg.in_channels * cfg.img_size**2, cfg.num_classes)), "smoke_test")
print("Checkpoint reloaded from epoch", ckpt["epoch"])

In [ ]:
# Clean up smoke-test artifacts
for p in [CHECKPOINT_DIR / "smoke_test.pt", RESULTS_DIR / "smoke_test_history.json"]:
    p.unlink(missing_ok=True)